# Practical Work 3

## Loading the packages

In [ ]:
import numpy as np
import matplotlib.pyplot as pl ## hello

## The Dataset


In [ ]:
%pip install ipympl

In [ ]:
import pandas as pd

 # Change this to any folder where data is if needed
mouse1 = pd.read_csv('EEG_mouse_data_1.csv')
mouse2 = pd.read_csv('EEG_mouse_data_2.csv')
dataset = pd.concat([mouse1, mouse2])
dataset

<font color="red">**For it to work on Colab, you will need to reload your session (Exécution -> redémarrer la session)**</font>

<font color="orange">**Make sure to put a large amount of points otherwise the cross validation folds will be really small**</font>

## Show the dataset

In [ ]:
dataset = np.array(dataset)
dataset

In [ ]:
input_data = dataset[:, 1:11].astype(float)
input_data = (input_data - input_data.mean(axis=0)) / (input_data.std(axis=0) + 1e-8)  # Standardize each feature
state_to_class = {"w": 0, "r": 1, "n": 2}
output_data = np.array([state_to_class[state] for state in dataset[:, 0]], dtype=int)

In [ ]:
input_data

In [ ]:
output_data

In [ ]:
# Check data balance
class_names = ["Awake", "REM", "Non-REM"]
unique, counts = np.unique(output_data, return_counts=True)
for label, count in zip(unique, counts):
    state = class_names[label]
    print(f"{state}: {count} samples ({count/len(output_data)*100:.1f}%)")

In [ ]:
%matplotlib inline

In [ ]:
import keras
from keras import layers
from sklearn.model_selection import StratifiedKFold

pl.clf()

keras.utils.set_random_seed(123)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=123)

class_colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
validation_color = "lightgray"
class_names = ["Awake", "REM", "Non-REM"]

for i, (train_index, test_index) in enumerate(skf.split(input_data, output_data)):
  pl.figure(figsize=(4,4))

  # Plot train data
  pl.scatter(input_data[train_index][:,0], input_data[train_index][:,1], c=[class_colors[int(d)] for d in output_data[train_index]], s=100)
  pl.scatter(input_data[test_index][:,0], input_data[test_index][:,1], c=validation_color, s=100)
  pl.title(f'Split {i + 1}, validation fold in gray.')
  pl.show()

In [ ]:
MODEL_CONFIG = {
  "input_dim": 10,
  "hidden_units": [32],
  "learning_rate": 5e-4,
  "batch_size": 128,
  "epochs": 300,
  "early_stopping_patience": 15,
  "reduce_lr_patience": 5,
  "reduce_lr_factor": 0.5,
  "min_lr": 1e-5,
}

def create_model(config=MODEL_CONFIG):
  # Update MODEL_CONFIG here after cross-validation to reuse in partie 3
  mlp = keras.Sequential()
  mlp.add(layers.Input(shape=(config["input_dim"],)))

  for units in config["hidden_units"]:
      mlp.add(layers.Dense(units, activation="relu"))

  mlp.add(layers.Dense(3, activation="softmax"))

  mlp.compile(
      optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"],
  )

  return mlp

mlp = create_model()
print("MODEL_CONFIG:", MODEL_CONFIG)
mlp.summary()

In [ ]:
history_list = []
trained_mlp = []

from sklearn.utils.class_weight import compute_class_weight
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=MODEL_CONFIG["early_stopping_patience"],
    min_delta=1e-4,
    restore_best_weights=True,
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=MODEL_CONFIG["reduce_lr_factor"],
    patience=MODEL_CONFIG["reduce_lr_patience"],
    min_delta=1e-4,
    min_lr=MODEL_CONFIG["min_lr"],
    verbose=1,
)

for i, (train_index, test_index) in enumerate(skf.split(input_data, output_data)):
  # We need to create a new model every time otherwise fit will continue previous training
  mlp = create_model()

  class_values = np.unique(output_data[train_index])
  class_weights = compute_class_weight(class_weight="balanced", classes=class_values, y=output_data[train_index])
  class_weights = np.sqrt(class_weights)
  class_weights = class_weights / np.mean(class_weights)
  class_weight_dict = dict(zip(class_values, class_weights))

  history = mlp.fit(
      x=input_data[train_index],
      y=output_data[train_index],
      validation_data=(input_data[test_index], output_data[test_index]),
      epochs=MODEL_CONFIG["epochs"],
      batch_size=MODEL_CONFIG["batch_size"],
      class_weight=class_weight_dict,
      callbacks=[early_stopping, reduce_lr],
  )

  history_list.append(history)
  trained_mlp.append(mlp)

# Plot training history

In [ ]:
def pad_history_values(histories, key):
    max_len = max(len(history.history[key]) for history in histories)
    padded_values = np.full((len(histories), max_len), np.nan)
    for row_index, history in enumerate(histories):
        values = np.asarray(history.history[key], dtype=float)
        padded_values[row_index, :len(values)] = values
    return padded_values

train_losses = pad_history_values(history_list, 'loss')
val_losses = pad_history_values(history_list, 'val_loss')

# Calculate mean and standard deviation for training and validation losses
mean_train_loss = np.nanmean(train_losses, axis=0)
std_train_loss = np.nanstd(train_losses, axis=0)
mean_val_loss = np.nanmean(val_losses, axis=0)
std_val_loss = np.nanstd(val_losses, axis=0)

epochs = np.arange(1, len(mean_train_loss) + 1)

# Plot mean and standard deviation for training loss
pl.plot(epochs, mean_train_loss, label='Training Loss (Mean)')
pl.fill_between(epochs, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, alpha=0.3, label='Training Loss (Std)')

# Plot mean and standard deviation for validation loss
pl.plot(epochs, mean_val_loss, label='Validation Loss (Mean)')
pl.fill_between(epochs, mean_val_loss - std_val_loss, mean_val_loss + std_val_loss, alpha=0.3, label='Validation Loss (Std)')

# Add labels and legend
pl.xlabel('Epochs')
pl.ylabel('Loss')
pl.legend()

# Display the plot
pl.show()

# Performances

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns

class_names = ["Awake", "REM", "Non-REM"]

def plot_confusion_matrix(confusion_matrix, title):
    # Plot confusion matrix
    pl.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix.astype(int), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    pl.title(title)
    pl.xlabel('Predicted')
    pl.ylabel('True')
    pl.show()

f1_scores = []
mean_confusion_matrix = np.zeros((3, 3))

for i, (train_index, test_index) in enumerate(skf.split(input_data, output_data)):
    # Evaluate the trained model on the test fold
    predictions = np.argmax(trained_mlp[i].predict(input_data[test_index]), axis=1)
    true_labels = output_data[test_index]

    # Compute confusion matrix
    cm = confusion_matrix(true_labels, predictions, labels=[0, 1, 2])
    mean_confusion_matrix += cm

    # Compute confusion matrix and plot
    plot_confusion_matrix(cm, f'Confusion Matrix - Fold {i + 1}')

    # Compute F1 score
    f1 = f1_score(true_labels, predictions, average='micro')
    f1_scores.append(f1)
    print(f"F1 Score - Fold {i + 1}: {f1}")

# Plot mean confusion matrix
plot_confusion_matrix(mean_confusion_matrix, 'Global confusion matrix')

# Calculate and display the mean F1 score across all folds
mean_f1_score = np.mean(f1_scores)
print(f"Mean F1 Score across all folds: {mean_f1_score}")

# Exercise

Please try changing hyperparameters (number of neurons, number of layers, learning rate, momentum, number of epochs...) and observe the impact it has on training and validation loss, convergence, and computation time. For instance, observe if there's overfitting if you put a high number (i.e. 128) of neurons in the hidden layer.

You can also experiment with different datasets (clear separation between classes, unbalanced...)
